In [8]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import sys, os

sys.path.append(os.environ['LP'])
import project
from project.core.utils import pprint

sys.path.append('../../../param_search')
import param_search as ps

ps.set_verbose(False)
ps.set_backend('slurm')

In [10]:
data_root = '/ocean/projects/asc170022p/mtragoza/lung-project/data/COPDGene'

In [11]:
import project.datasets.copdgene
dataset_cls = project.datasets.copdgene.COPDGeneDataset
ds = dataset_cls(data_root)
ds

project.datasets.copdgene.COPDGeneDataset('/ocean/projects/asc170022p/mtragoza/lung-project/data/COPDGene')

In [12]:
ds.load_metadata()

In [13]:
examples = ds.list_examples(
    subjects='../../data/COPDGene/sample1000_2025-05-21.csv',
    state_pairs=[('EXP', 'INSP')]
)
len(examples)

1000

In [14]:
base_dir = '2026-05-27_preprocess'

template = '''\
#!/bin/bash -l
#SBATCH --job-name={job_name}
#SBATCH --account=asc170022p
#SBATCH --partition=GPU-shared
#SBATCH --gres=gpu:1
#SBATCH -t 6:00:00
set -eo pipefail

LP=$PROJECT/lung-project
NB=$PROJECT/lung-project/notebooks/copdgene

mamba activate /ocean/projects/asc170022p/mtragoza/mambaforge/envs/warp

ln -s network_weights/gradicon_lung1.0/Step_2_final.trch

python $LP/scripts/preprocess.py {config} \\
    --set dataset.name={data_name} \\
    --set dataset.root={data_root} \\
    --set dataset.examples.subjects={subject} \\
    --set dataset.examples.variant={variant} \\

'''
name_format = '{params_hash}'

grid = ps.param_grid(
    config='2026-05-27_config.yaml',
    data_name='COPDGene',
    data_root=data_root,
    subject=[ex.subject for ex in examples],
    variant='2026-05-27'
)
len(grid)

1000

In [15]:
%autoreload
try:
    jobs = ps.setup(base_dir, template, name_format, grid, overwrite=False)
except OSError:
    jobs = ps.load(base_dir)

jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,3a1ddc52cf1bce93,COMPLETED,1,41049497,v005,00:01:36,| ├── 'relative_loss': True\n | └──...,,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,16514P,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,None,None
1,7eee1563373873d7,COMPLETED,1,41049498,v029,00:10:29,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,20748Q,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,None,None
2,5f1ffa673968c6dd,COMPLETED,1,41049499,v031,00:09:49,Reindexing cell labels\nRemoving background ce...,Total optimization time: 24.3364s\n\nRunning s...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,11007Z,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,None,None
3,0a7e3c5324f4ac8b,COMPLETED,1,41049500,v005,00:07:29,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,14771Z,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,None,None
4,b99621050312ac4e,COMPLETED,1,41049501,v029,00:04:28,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,13651K,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,e8676ea45de24d49,PENDING,1,41050492,(Priority),0:00,None,None,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,20519B,2026-05-27,NaN,2026-05-28T16:01:53,status,None,None,False,None,None
996,3db2c84d0afc5372,PENDING,1,41050493,(Priority),0:00,None,None,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,12294H,2026-05-27,NaN,2026-05-28T16:01:53,status,None,None,False,None,None
997,c4a93ce51e790114,PENDING,1,41050494,(Priority),0:00,None,None,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,23123R,2026-05-27,NaN,2026-05-28T16:01:53,status,None,None,False,None,None
998,d50289a79d0250f0,PENDING,1,41050495,(Priority),0:00,None,None,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,16546C,2026-05-27,NaN,2026-05-28T16:01:53,status,None,None,False,None,None


In [16]:
%autoreload
jobs = ps.recover(jobs)
jobs = ps.status(jobs)
jobs = ps.history(jobs)

jobs.groupby(['job_state']).count()

,job_name,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,script_path,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
job_state,,,,,,,,,,,,,,,,,,,,,
COMPLETED,44,44,44,44,44,44,44,44,44,44,...,44,44,0,19,44,44,44,44,0,0
FAILED,956,956,956,956,956,0,0,956,956,956,...,956,956,0,956,956,956,956,956,0,0


In [17]:
jobs = ps.collect(jobs)

In [22]:
query_jobs = jobs[jobs.job_state == 'FAILED']
query_jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
44,1778d83f25c8d744,FAILED,1,41049541,v024,00:04:02,Input 1-connected components: 6\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,25130Y,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
45,d35bf1d5100877bd,FAILED,1,41049542,v018,00:01:51,Input 1-connected components: 2\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,15623P,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
46,dfb0b9957e28d7c0,FAILED,1,41049543,v018,00:01:51,Input 1-connected components: 1\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,19027T,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
47,964f14fad7f51ffc,FAILED,1,41049544,v008,00:02:24,Input 1-connected components: 2\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,14380K,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
48,a897308d388bfbcf,FAILED,1,41049545,v008,00:01:40,Input 1-connected components: 1\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,10212V,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,e8676ea45de24d49,FAILED,1,41050492,v008,00:01:47,Input 1-connected components: 7\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,20519B,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
996,3db2c84d0afc5372,FAILED,1,41050493,w004,00:01:37,Input 1-connected components: 1\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,12294H,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
997,c4a93ce51e790114,FAILED,1,41050494,v023,00:02:07,Input 1-connected components: 1\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,23123R,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
998,d50289a79d0250f0,FAILED,1,41050495,v024,00:01:57,Input 1-connected components: 1\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,16546C,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>


In [21]:
query_jobs.iloc[0]

job_name                                             1778d83f25c8d744
job_state                                                      FAILED
n_submits                                                           1
job_id                                                       41049541
node_id                                                          v024
runtime                                                      00:04:02
stdout              Input 1-connected components: 6\n  Voxel count...
stderr                  result = preprocessing.api.preprocess_exam...
base_dir            /ocean/projects/asc170022p/mtragoza/lung-proje...
work_dir            /ocean/projects/asc170022p/mtragoza/lung-proje...
script_path         /ocean/projects/asc170022p/mtragoza/lung-proje...
output_path         /ocean/projects/asc170022p/mtragoza/lung-proje...
log_dir             /ocean/projects/asc170022p/mtragoza/lung-proje...
stdout_path         /ocean/projects/asc170022p/mtragoza/lung-proje...
stderr_path         

In [20]:
print(query_jobs.iloc[0].stderr)

    result = preprocessing.api.preprocess_example(ex, config)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/ocean/projects/asc170022p/mtragoza/lung-project/project/preprocessing/api.py", line 11, in preprocess_example
    return pipelines.preprocess_copdgene(ex, config)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/ocean/projects/asc170022p/mtragoza/lung-project/project/preprocessing/pipelines.py", line 154, in preprocess_copdgene
    _ensure_output( # disp_field
  File "/ocean/projects/asc170022p/mtragoza/lung-project/project/preprocessing/pipelines.py", line 27, in _ensure_output
    return True, func(*args, **kwargs)
                 ^^^^^^^^^^^^^^^^^^^^^
  File "/ocean/projects/asc170022p/mtragoza/lung-project/project/preprocessing/stages.py", line 454, in register_displacement_field
    registration.run_unigradicon_registration(
  File "/ocean/projects/asc170022p/mtragoza/lung-project/project/preprocessing/registration.py", line 53, in run_

In [24]:
import pandas as pd
jobs.loc[jobs.job_state == 'FAILED', 'job_id'] = pd.NA

In [25]:
jobs = ps.submit(jobs)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,3a1ddc52cf1bce93,COMPLETED,1,41049497,v005,00:01:36,| ├── 'relative_loss': True\n | └──...,,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,16514P,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,<NA>,<NA>
1,7eee1563373873d7,COMPLETED,1,41049498,v029,00:10:29,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,20748Q,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,<NA>,<NA>
2,5f1ffa673968c6dd,COMPLETED,1,41049499,v031,00:09:49,Reindexing cell labels\nRemoving background ce...,Total optimization time: 24.3364s\n\nRunning s...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,11007Z,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,<NA>,<NA>
3,0a7e3c5324f4ac8b,COMPLETED,1,41049500,v005,00:07:29,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,14771Z,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,<NA>,<NA>
4,b99621050312ac4e,COMPLETED,1,41049501,v029,00:04:28,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,13651K,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,e8676ea45de24d49,SUBMITTED,2,41101544,v008,00:01:47,Input 1-connected components: 7\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,20519B,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
996,3db2c84d0afc5372,SUBMITTED,2,41101545,w004,00:01:37,Input 1-connected components: 1\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,12294H,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
997,c4a93ce51e790114,SUBMITTED,2,41101546,v023,00:02:07,Input 1-connected components: 1\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,23123R,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
998,d50289a79d0250f0,SUBMITTED,2,41101547,v024,00:01:57,Input 1-connected components: 1\n Voxel count...,result = preprocessing.api.preprocess_exam...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,16546C,2026-05-27,NaN,2026-05-28T16:01:53,history,True,2026-06-01T12:18:59,False,<NA>,<NA>
